# Exploratory data analysis

Descriptive analysis of the annotated Bayes-tutor dataset used by the BKT models. The loader excludes Q0 warm-up rows. Blank KC and error-flag cells are structural: they indicate that the KC or flag was not observed for that response.

## 1. Load data

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from scripts.data import FLAG_COLS, KC_COLS, load_data


DATA_PATH = Path("data/data_annotated.csv")
data = load_data(DATA_PATH)
data.head()

,participant_id,question_number,question_text,transcript,designed_kcs,adaptive_kcs,question_correct,kc1_sample_space,kc2_conditioning,kc3_joint_chain,kc4_total_probability,kc5_bayes_update,conjunction,inverse,time_axis,denominator_neglect,base_rate_neglect,annotation_rationale
0,P01,1,A panel of psychologists interviewed 30 nurses...,I decided to use Bayes' Theorem to solve this ...,"[kc1_sample_space, kc5_bayes_update]","[kc1_sample_space, kc5_bayes_update]",wrong,correct,,,,wrong,,,,,quiet,"Attempted a genuine Bayes revision, prior 0.3 ..."
1,P02,1,A panel of psychologists interviewed 30 nurses...,"Well, if we ignore the description, the probab...","[kc1_sample_space, kc5_bayes_update]","[kc1_sample_space, kc5_bayes_update]",correct,correct,,,,correct,,,,,quiet,Holds thirty percent from the split after expl...
2,P03,1,A panel of psychologists interviewed 30 nurses...,"Oh, I think there's a 30% chance that Sam is o...","[kc1_sample_space, kc5_bayes_update]","[kc1_sample_space, kc5_bayes_update]",correct,correct,,,,correct,,,,,,Thirty percent with the description evaluated ...
3,P04,1,A panel of psychologists interviewed 30 nurses...,"So, the answer is 0.03, no, sorry, the answer ...","[kc1_sample_space, kc5_bayes_update]",[kc1_sample_space],correct,correct,,,,,,,,,,"Bare sampling read, thirty of one hundred, the..."
4,P05,1,A panel of psychologists interviewed 30 nurses...,"The answer should be 30 over 100, because 100 ...","[kc1_sample_space, kc5_bayes_update]",[kc1_sample_space],correct,correct,,,,,,,,,,"Bare sampling read, kc5 not engaged, kc1 corre..."


## 2. Dataset structure and quality

In [2]:
participant_sizes = data.groupby("participant_id").size()
question_sizes = data.groupby("question_number").size()

overview = pd.Series(
    {
        "rows": len(data),
        "columns": data.shape[1],
        "participants": data["participant_id"].nunique(),
        "questions": data["question_number"].nunique(),
        "rows per participant": participant_sizes.iloc[0] if participant_sizes.nunique() == 1 else "varies",
        "participants per question": question_sizes.iloc[0] if question_sizes.nunique() == 1 else "varies",
    },
    name="value",
).to_frame()

quality_checks = pd.Series(
    {
        "duplicate participant-question rows": data.duplicated(["participant_id", "question_number"]).sum(),
        "invalid question-correct labels": (~data["question_correct"].isin(["correct", "wrong"])).sum(),
        "participants without 12 rows": participant_sizes.ne(12).sum(),
        "questions without 26 participants": question_sizes.ne(26).sum(),
        "blank question texts": data["question_text"].eq("").sum(),
        "blank transcripts": data["transcript"].eq("").sum(),
        "blank annotation rationales": data["annotation_rationale"].eq("").sum(),
    },
    name="count",
).to_frame()

display(overview)
display(quality_checks)

,value
rows,312
columns,18
participants,26
questions,12
rows per participant,12
participants per question,26


,count
duplicate participant-question rows,0
invalid question-correct labels,0
participants without 12 rows,0
questions without 26 participants,0
blank question texts,0
blank transcripts,0
blank annotation rationales,0


## 3. Question-correctness balance

The majority-class baseline is useful when interpreting model accuracy and F1.

In [3]:
qc_balance = (
    data["question_correct"].value_counts()
    .reindex(["correct", "wrong"], fill_value=0)
    .rename_axis("label")
    .to_frame("n")
)
qc_balance["rate"] = qc_balance["n"] / qc_balance["n"].sum()
qc_balance["distribution"] = qc_balance["rate"].map(lambda value: "█" * round(value * 30))

correct_rate = qc_balance.loc["correct", "rate"]
majority_baseline = pd.Series(
    {
        "always-correct accuracy": correct_rate,
        "always-correct F1": 2 * correct_rate / (1 + correct_rate),
        "correct-to-wrong ratio": qc_balance.loc["correct", "n"] / qc_balance.loc["wrong", "n"],
    },
    name="value",
).to_frame()

display(qc_balance)
display(majority_baseline)

,n,rate,distribution
label,,,
correct,200,0.641026,███████████████████
wrong,112,0.358974,███████████


,value
always-correct accuracy,0.641026
always-correct F1,0.781250
correct-to-wrong ratio,1.785714


## 4. Performance by participant

In [4]:
participant_summary = (
    data.assign(is_correct=data["question_correct"].eq("correct").astype(int))
    .groupby("participant_id")
    .agg(questions=("question_number", "size"), correct=("is_correct", "sum"))
)
participant_summary["wrong"] = participant_summary["questions"] - participant_summary["correct"]
participant_summary["correct_rate"] = participant_summary["correct"] / participant_summary["questions"]
participant_summary["distribution"] = participant_summary["correct_rate"].map(lambda value: "█" * round(value * 20))
participant_summary.sort_values(["correct_rate", "participant_id"])

,questions,correct,wrong,correct_rate,distribution
participant_id,,,,,
P15,12,2,10,0.166667,███
P16,12,2,10,0.166667,███
P01,12,3,9,0.250000,█████
P03,12,3,9,0.250000,█████
P23,12,3,9,0.250000,█████
P13,12,4,8,0.333333,███████
P20,12,6,6,0.500000,██████████
P26,12,6,6,0.500000,██████████
P06,12,7,5,0.583333,████████████


## 5. Performance by question

Question-level correctness mixes item difficulty with its required KC structure.

In [5]:
question_summary = (
    data.assign(is_correct=data["question_correct"].eq("correct").astype(int))
    .groupby("question_number")
    .agg(responses=("participant_id", "size"), correct=("is_correct", "sum"))
)
question_summary["wrong"] = question_summary["responses"] - question_summary["correct"]
question_summary["correct_rate"] = question_summary["correct"] / question_summary["responses"]
question_summary["designed_kcs"] = data.groupby("question_number")["designed_kcs"].first().map(lambda values: "; ".join(values))
question_summary["n_designed_kcs"] = data.groupby("question_number")["designed_kcs"].first().map(len)
question_summary["distribution"] = question_summary["correct_rate"].map(lambda value: "█" * round(value * 20))
question_summary

,responses,correct,wrong,correct_rate,designed_kcs,n_designed_kcs,distribution
question_number,,,,,,,
1,26,22,4,0.846154,kc1_sample_space; kc5_bayes_update,2,█████████████████
2,26,21,5,0.807692,kc1_sample_space,1,████████████████
3,26,25,1,0.961538,kc2_conditioning,1,███████████████████
4,26,13,13,0.500000,kc2_conditioning,1,██████████
5,26,14,12,0.538462,kc2_conditioning,1,███████████
6,26,9,17,0.346154,kc2_conditioning; kc3_joint_chain,2,███████
7,26,24,2,0.923077,kc3_joint_chain; kc4_total_probability,2,██████████████████
8,26,14,12,0.538462,kc3_joint_chain; kc4_total_probability; kc5_ba...,3,███████████
9,26,17,9,0.653846,kc5_bayes_update,1,█████████████


## 6. KC coverage and cell correctness

A realized cell is one explicitly annotated as `correct` or `wrong`.

In [6]:
kc_rows = []
for kc in KC_COLS:
    realized = data[kc].isin(["correct", "wrong"])
    correct = data[kc].eq("correct")
    wrong = data[kc].eq("wrong")
    kc_rows.append(
        {
            "kc": kc,
            "designed_mentions": data["designed_kcs"].map(lambda values: kc in values).sum(),
            "adaptive_mentions": data["adaptive_kcs"].map(lambda values: kc in values).sum(),
            "realized_cells": realized.sum(),
            "cell_correct": correct.sum(),
            "cell_wrong": wrong.sum(),
            "cell_correct_rate": correct.sum() / realized.sum(),
        }
    )

kc_summary = pd.DataFrame(kc_rows).set_index("kc")
kc_summary["distribution"] = kc_summary["cell_correct_rate"].map(lambda value: "█" * round(value * 20))
kc_summary

,designed_mentions,adaptive_mentions,realized_cells,cell_correct,cell_wrong,cell_correct_rate,distribution
kc,,,,,,,
kc1_sample_space,130,152,152,143,9,0.940789,███████████████████
kc2_conditioning,182,156,156,133,23,0.852564,█████████████████
kc3_joint_chain,156,165,165,129,36,0.781818,████████████████
kc4_total_probability,130,132,132,108,24,0.818182,████████████████
kc5_bayes_update,156,163,163,129,34,0.791411,████████████████


## 7. Cell correctness versus question correctness

This compares the final-answer success rate when each realized KC cell was correct versus wrong. It is descriptive association, not a causal effect.

In [7]:
association_rows = []
for kc in KC_COLS:
    cell_correct = data[data[kc].eq("correct")]
    cell_wrong = data[data[kc].eq("wrong")]
    rate_if_correct = cell_correct["question_correct"].eq("correct").mean()
    rate_if_wrong = cell_wrong["question_correct"].eq("correct").mean()
    association_rows.append(
        {
            "kc": kc,
            "n_cell_correct": len(cell_correct),
            "qc_rate_if_cell_correct": rate_if_correct,
            "n_cell_wrong": len(cell_wrong),
            "qc_rate_if_cell_wrong": rate_if_wrong,
            "rate_difference": rate_if_correct - rate_if_wrong,
        }
    )

kc_qc_association = pd.DataFrame(association_rows).set_index("kc")
kc_qc_association

,n_cell_correct,qc_rate_if_cell_correct,n_cell_wrong,qc_rate_if_cell_wrong,rate_difference
kc,,,,,
kc1_sample_space,143,0.678322,9,0.111111,0.567211
kc2_conditioning,133,0.751880,23,0.043478,0.708401
kc3_joint_chain,129,0.767442,36,0.111111,0.656331
kc4_total_probability,108,0.861111,24,0.000000,0.861111
kc5_bayes_update,129,0.829457,34,0.000000,0.829457


## 8. Designed versus adaptive KC sets

In [8]:
kc_alignment = data[["participant_id", "question_number"]].copy()
kc_alignment["designed_count"] = data["designed_kcs"].map(len)
kc_alignment["adaptive_count"] = data["adaptive_kcs"].map(len)
kc_alignment["overlap_count"] = [len(set(d) & set(a)) for d, a in zip(data["designed_kcs"], data["adaptive_kcs"])]
kc_alignment["missing_designed"] = [len(set(d) - set(a)) for d, a in zip(data["designed_kcs"], data["adaptive_kcs"])]
kc_alignment["additional_adaptive"] = [len(set(a) - set(d)) for d, a in zip(data["designed_kcs"], data["adaptive_kcs"])]
kc_alignment["exact_match"] = [set(d) == set(a) for d, a in zip(data["designed_kcs"], data["adaptive_kcs"])]

alignment_overview = pd.Series(
    {
        "exact set matches": kc_alignment["exact_match"].sum(),
        "exact match rate": kc_alignment["exact_match"].mean(),
        "rows missing at least one designed KC": kc_alignment["missing_designed"].gt(0).sum(),
        "rows with an additional adaptive KC": kc_alignment["additional_adaptive"].gt(0).sum(),
        "rows with no adaptive KC": kc_alignment["adaptive_count"].eq(0).sum(),
    },
    name="value",
).to_frame()

alignment_by_question = kc_alignment.groupby("question_number").agg(
    mean_designed=("designed_count", "mean"),
    mean_adaptive=("adaptive_count", "mean"),
    exact_match_rate=("exact_match", "mean"),
    mean_missing_designed=("missing_designed", "mean"),
    mean_additional_adaptive=("additional_adaptive", "mean"),
)

display(alignment_overview)
display(alignment_by_question)

,value
exact set matches,175.000000
exact match rate,0.560897
rows missing at least one designed KC,69.000000
rows with an additional adaptive KC,79.000000
rows with no adaptive KC,5.000000


,mean_designed,mean_adaptive,exact_match_rate,mean_missing_designed,mean_additional_adaptive
question_number,,,,,
1,2.0,1.538462,0.538462,0.461538,0.000000
2,1.0,1.461538,0.615385,0.000000,0.461538
3,1.0,1.269231,0.692308,0.076923,0.346154
4,1.0,2.769231,0.115385,0.038462,1.807692
5,1.0,1.961538,0.038462,0.192308,1.153846
6,2.0,2.192308,0.423077,0.346154,0.538462
7,2.0,1.961538,0.961538,0.038462,0.000000
8,3.0,2.653846,0.653846,0.384615,0.038462
9,1.0,1.038462,0.769231,0.153846,0.192308


## 9. Error-flag prevalence

Flag rates use `quiet + fired` as the annotated denominator; blank rows are treated as outside that flag's evaluated context.

In [9]:
flag_rows = []
for flag in FLAG_COLS:
    counts = data[flag].value_counts()
    quiet = counts.get("quiet", 0)
    fired = counts.get("fired", 0)
    evaluated = quiet + fired
    fired_rows = data[data[flag].eq("fired")]
    flag_rows.append(
        {
            "flag": flag,
            "blank": counts.get("", 0),
            "quiet": quiet,
            "fired": fired,
            "fire_rate_when_evaluated": fired / evaluated if evaluated else float("nan"),
            "qc_correct_rate_when_fired": fired_rows["question_correct"].eq("correct").mean() if fired else float("nan"),
        }
    )

flag_summary = pd.DataFrame(flag_rows).set_index("flag")
flag_summary

,blank,quiet,fired,fire_rate_when_evaluated,qc_correct_rate_when_fired
flag,,,,,
conjunction,290,22,0,0.000000,NaN
inverse,286,25,1,0.038462,0.0
time_axis,278,28,6,0.176471,0.0
denominator_neglect,221,88,3,0.032967,0.0
base_rate_neglect,191,114,7,0.057851,0.0


## 10. Reproducible headline findings

In [10]:
hardest_rate = question_summary["correct_rate"].min()
easiest_rate = question_summary["correct_rate"].max()
hardest_questions = question_summary.index[question_summary["correct_rate"].eq(hardest_rate)].tolist()
easiest_questions = question_summary.index[question_summary["correct_rate"].eq(easiest_rate)].tolist()

headline_findings = pd.Series(
    {
        "question-correct rate": data["question_correct"].eq("correct").mean(),
        "participant correct-rate range": f"{participant_summary['correct_rate'].min():.1%}–{participant_summary['correct_rate'].max():.1%}",
        "hardest question(s)": f"{hardest_questions} ({hardest_rate:.1%} correct)",
        "easiest question(s)": f"{easiest_questions} ({easiest_rate:.1%} correct)",
        "lowest cell-correct KC": kc_summary["cell_correct_rate"].idxmin(),
        "highest cell-correct KC": kc_summary["cell_correct_rate"].idxmax(),
        "designed/adaptive exact-match rate": kc_alignment["exact_match"].mean(),
        "total fired error flags": int(sum(data[flag].eq("fired").sum() for flag in FLAG_COLS)),
    },
    name="value",
).to_frame()
headline_findings

,value
question-correct rate,0.641026
participant correct-rate range,16.7%–100.0%
hardest question(s),[11] (26.9% correct)
easiest question(s),[3] (96.2% correct)
lowest cell-correct KC,kc3_joint_chain
highest cell-correct KC,kc1_sample_space
designed/adaptive exact-match rate,0.560897
total fired error flags,17
